# Mobile Game Lookalike Audience Modeling with KNN for User Acquisition

## Introduction

This project explores how K-Nearest Neighbors (KNN) can be used to build lookalike audiences for mobile game user acquisition (UA). High-value players are identified using historical revenue, and KNN is applied to find other players with similar behavioral profiles.

The resulting audience is compared with the remaining player population to assess differences in historical monetization. The goal is to explore whether behavioral lookalikes could be useful for UA targeting, using synthetic data.

In [1]:
import pandas as pd
import numpy as np

# Load the synthetic player behavior dataset
results_df = pd.read_csv('synthetic_users.csv')

# Keep player identifiers for interpretation; exclude segment from model features
results_df = results_df.drop(columns=['segment'], errors='ignore')

# Display the dataset
results_df

,player_id,revenue,feature_A,feature_B,feature_C,f_neutral_1,f_neutral_2,f_neutral_3
0,0,0.112984,231.0,1482.0,1773.0,0.797903,0.159682,0.298777
1,1,1.853651,69.0,1682.0,1684.0,0.959326,0.745312,0.342203
2,2,1.217135,261.0,1178.0,1367.0,0.518735,0.072726,0.174626
3,3,0.999441,137.0,1338.0,1095.0,0.302235,0.880557,0.705730
4,4,1.303194,109.0,1650.0,547.0,0.981310,0.525359,0.394562
...,...,...,...,...,...,...,...,...
9995,9995,1.072332,75.0,1495.0,819.0,0.524016,0.649930,0.242501
9996,9996,2.378438,98.0,953.0,316.0,0.596799,0.206719,0.997439
9997,9997,0.954210,202.0,263.0,145.0,0.788424,0.576036,0.139574
9998,9998,1.304638,44.0,1951.0,1586.0,0.804460,0.451873,0.582693


In [2]:
df = results_df.copy()
whale_threshold = df['revenue'].quantile(0.98)

conditions = [
    df['revenue'] >= whale_threshold
]

choices = ['Whale']

df['segment'] = np.select(conditions, choices, default='Normal')

df['segment'].value_counts()


segment
Normal    9800
Whale      200
Name: count, dtype: int64

In [3]:
summary = df.groupby('segment')['revenue'].agg(
    min='min',
    p25=lambda x: x.quantile(0.25),
    median='median',
    p75=lambda x: x.quantile(0.75),
    max='max',
    mean='mean',
    count='count'
)

summary.round(2)

,min,p25,median,p75,max,mean,count
segment,,,,,,,
Normal,0.01,0.51,0.91,1.52,24.73,1.62,9800
Whale,24.77,28.88,32.70,39.60,68.11,35.13,200


In [4]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np

features = [
    'feature_A',
    'feature_B',
    'feature_C',
    'f_neutral_1',
    'f_neutral_2',
    'f_neutral_3'
]

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(df[features])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

knn = NearestNeighbors(n_neighbors=8, metric='euclidean')
knn.fit(X_scaled)

whale_idx = df.index[df['segment'] == 'Whale'].to_numpy()

distances, indices = knn.kneighbors(X_scaled[whale_idx])

neighbor_idx = np.unique(indices.flatten())
neighbor_idx = np.setdiff1d(neighbor_idx, whale_idx)

df['knn_neighbors_only'] = 0
df.loc[neighbor_idx, 'knn_neighbors_only'] = 1

In [7]:
standard_normal = df[
    (df['segment'] != 'Whale') &
    (df['knn_neighbors_only'] == 0)
]

knn_neighbors = df[df['knn_neighbors_only'] == 1]

whales = df[df['segment'] == 'Whale']

comparison = pd.DataFrame({
    'Standard Players': standard_normal['revenue'].agg(['count', 'mean', 'median']),
    'KNN Lookalikes': knn_neighbors['revenue'].agg(['count', 'mean', 'median']),
    'High-Value Seeds': whales['revenue'].agg(['count', 'mean', 'median'])
})

comparison.round(2)

,Standard Players,KNN Lookalikes,High-Value Seeds
count,9306.00,494.00,200.00
mean,1.23,8.97,35.13
median,0.89,2.88,32.70


## Conclusion

The KNN lookalike audience showed higher historical revenue than the remaining player population, with a median revenue of 2.88 compared with 0.89 for standard players.

The identified lookalikes could serve as a candidate audience for targeted user acquisition campaigns. However, further evaluation using future player data or a controlled UA experiment would be needed to determine whether targeting users with similar behavioral profiles leads to better acquisition outcomes.

The analysis uses synthetic data and provides an exploratory approach to lookalike audience creation.